# Step 0 — Sector definition and validation

## Scope
Precision agriculture applied to crop production.
Livestock and aquaculture are outside our scope.

## Initial search strategy
A patent family is selected if at least one application:
- has CPC code A01B79/005; OR
- matches both an agricultural term and a technological term
  across its title and abstract.

The keyword lists are adapted from Tey et al., Table 1.
This is an initial search strategy to be evaluated through validation.

## 1. Load the search configuration

We read the sector definition, CPC prefixes and keyword lists from
`config/sector.yaml`.

In [1]:
from pathlib import Path
import yaml

# Locate the repository from either its root or notebooks folder.
current_dir = Path.cwd()

if (current_dir / "config" / "sector.yaml").is_file():
    repo_dir = current_dir
elif (current_dir.parent / "config" / "sector.yaml").is_file():
    repo_dir = current_dir.parent
else:
    raise FileNotFoundError("Cannot find config/sector.yaml")

config_path = repo_dir / "config" / "sector.yaml"

with config_path.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

search = config["search"]

## 2. Connect to PATSTAT

In [2]:
from sqlalchemy import and_, or_, func

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS224_APPLN_CPC,
)

# Use TEST to check that the code works.
# Switch to PROD before drawing the actual validation sample.
#ENV = "TEST"
ENV = "PROD"

patstat = PatstatClient(env=ENV)
db = patstat.orm()

print("PATSTAT environment:", ENV)

PATSTAT environment: PROD


## 3. Define the search conditions

Our initial search has two alternative routes:

- **CPC route:** an application has a classification matching A01B79/005.
- **Keyword route:** an application contains at least one agricultural
  term AND at least one technological term across its title and abstract.

Within each keyword list, terms are connected by OR.
The two keyword lists are connected by AND.

The functions are adapted from the professor's notebook. The main change
is the separation of keywords into agricultural and technological groups.

Keyword matching uses text substrings, not whole words. This may produce
irrelevant matches, which we will investigate during validation.

In [3]:
AGRICULTURAL_TERMS = config["search"]["agricultural_terms"]
TECHNOLOGICAL_TERMS = config["search"]["technological_terms"]
SECTOR_CPC_PREFIXES = config["search"]["cpc_prefixes"]

if not all([
    AGRICULTURAL_TERMS,
    TECHNOLOGICAL_TERMS,
    SECTOR_CPC_PREFIXES,
]):
    raise ValueError("Keyword and CPC lists must not be empty.")


def keyword_condition(terms):
    """Match at least one expression in the title or abstract."""

    # Treat missing text as empty; ignore uppercase/lowercase.
    title = func.lower(
        func.coalesce(TLS202_APPLN_TITLE.appln_title, "")
    )
    abstract = func.lower(
        func.coalesce(TLS203_APPLN_ABSTR.appln_abstract, "")
    )

    # OR across the 14 expressions and both text fields.
    # Matches literal text, without expanding synonyms or hyphen variants.
    return or_(
        *(title.contains(t.lower(), autoescape=True) for t in terms),
        *(abstract.contains(t.lower(), autoescape=True) for t in terms),
    )


def symbol_prefix_condition(symbol_column, codes):
    """Match classification prefixes after removing spaces."""

    # Unchanged from the previous version.
    symbol = func.replace(symbol_column, " ", "")

    return or_(*(
        symbol.like(code.replace(" ", "") + "%")
        for code in codes
    ))


# Columns available when inspecting individual applications.
APPLN_COLUMNS = (
    TLS201_APPLN.docdb_family_id,
    TLS201_APPLN.appln_id,
    TLS201_APPLN.appln_auth,
    TLS201_APPLN.appln_nr,
    TLS201_APPLN.appln_kind,
    TLS201_APPLN.appln_filing_date,
)


def query_by_keywords(*columns):
    """WIPO-inspired search: agricultural AND positioning terms."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .outerjoin(
            TLS202_APPLN_TITLE,
            TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id,
        )
        .outerjoin(
            TLS203_APPLN_ABSTR,
            TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id,
        )
        # Require at least one term from each group.
        .filter(
            and_(
                keyword_condition(AGRICULTURAL_TERMS),
                keyword_condition(TECHNOLOGICAL_TERMS),
            )
        )
    )

def query_by_cpc(*columns):
    """Select applications matching the configured CPC prefixes."""

    # Unchanged: a CPC match does not require a keyword match.
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .join(
            TLS224_APPLN_CPC,
            TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id,
        )
        .filter(
            symbol_prefix_condition(
                TLS224_APPLN_CPC.cpc_class_symbol,
                SECTOR_CPC_PREFIXES,
            )
        )
    )


print("Query functions are ready.")

Query functions are ready.


## 4. Combine the search routes and identify families

We combine the CPC and keyword results using OR: either route is sufficient
for inclusion.

We retain patent-invention applications (`ipr_type = "PI"`) with a valid
DOCDB family identifier. We then select distinct family IDs using UNION.

A family is selected when at least one qualifying application matches
either search route. Families found through both routes are counted once.

In [4]:
family_id = TLS201_APPLN.docdb_family_id.label("docdb_family_id")

base_filters = (
    TLS201_APPLN.docdb_family_id > 0,
    TLS201_APPLN.ipr_type == "PI",
)

families_by_cpc = (
    query_by_cpc(family_id)
    .filter(*base_filters)
)

families_by_keywords = (
    query_by_keywords(family_id)
    .filter(*base_filters)
)

# CPC OR keywords; UNION removes duplicate family IDs.
sector_ids = families_by_cpc.union(families_by_keywords)

patstat.df(
    sector_ids
    .order_by("docdb_family_id")
    .limit(10)
)

,docdb_family_id
0,3774156
1,3798016
2,3828303
3,3828568
4,4021094
5,4021870
6,4083641
7,4107297
8,4150876
9,4164232


## 5. Apply the analysis period

We restrict the dataset to families dated between 2000 and 2022.

For each selected family, we calculate the minimum `earliest_filing_year`
across its patent-invention applications and apply the time filter to
that family-level value.

The end year is an initial choice intended to reduce exposure to recent
incomplete data. It does not guarantee complete coverage.

In [5]:
YEAR_MIN = config["analysis"]["year_min"]
YEAR_MAX = config["analysis"]["year_max"]
FAMILY_TIME_COLUMN = config["analysis"]["family_time_column"]

# Use the earliest available filing year across the family's
# patent-invention applications.
family_year = func.min(
    getattr(TLS201_APPLN, FAMILY_TIME_COLUMN)
)

sector_families_query = (
    db.query(
        TLS201_APPLN.docdb_family_id.label("docdb_family_id"),
        family_year.label("family_year"),
    )
    .filter(
        TLS201_APPLN.docdb_family_id.in_(
            sector_ids.subquery().select()
        ),
        TLS201_APPLN.ipr_type == "PI",
    )
    .group_by(TLS201_APPLN.docdb_family_id)

    # Apply the time window after calculating the family-level year.
    .having(family_year.between(YEAR_MIN, YEAR_MAX))
)

preview = patstat.df(
    sector_families_query
    .order_by(TLS201_APPLN.docdb_family_id)
    .limit(10)
)

preview

,docdb_family_id,family_year
0,3828303,2001
1,3828568,2001
2,4021094,2000
3,4021870,2000
4,4168004,2000
5,4538341,2001
6,4554232,2001
7,4565376,2000
8,4655480,2001
9,4665578,2001


In [6]:
# Count the selected families without downloading the full dataset.
family_scope = sector_families_query.subquery()

count_query = (
    db.query(func.count().label("n_families"))
    .select_from(family_scope)
)

family_count = patstat.df(count_query)
family_count

,n_families
0,4749


## 6. Identify how families entered the dataset

We divide the selected families into three mutually exclusive groups:

- **CPC only:** matched the selected CPC but not the keyword conditions.
- **Keywords only:** matched the keyword conditions but not the selected CPC.
- **Both CPC and keywords:** matched both search routes.

The three counts should sum to the total candidate dataset: 34,448 families
in the current run.

This comparison shows how much each route contributes. It does not yet
measure relevance. It will help us design the validation sample and
investigate which search conditions produce irrelevant results.

At family level, the CPC match and keyword match may come from different
applications within the same family.

In [7]:
import pandas as pd

# Use the same family population and time window as the total count.
scope = sector_families_query.subquery()
cpc_ids = families_by_cpc.subquery()
keyword_ids = families_by_keywords.subquery()

has_cpc = scope.c.docdb_family_id.in_(
    db.query(cpc_ids.c.docdb_family_id)
)
has_keywords = scope.c.docdb_family_id.in_(
    db.query(keyword_ids.c.docdb_family_id)
)

# Each family belongs to exactly one group.
groups = {
    "CPC only": and_(has_cpc, ~has_keywords),
    "Keywords only": and_(has_keywords, ~has_cpc),
    "Both CPC and keywords": and_(has_cpc, has_keywords),
}

rows = []

for group, condition in groups.items():
    query = (
        db.query(func.count().label("n_families"))
        .select_from(scope)
        .filter(condition)
    )

    count = int(patstat.df(query).iloc[0]["n_families"])
    rows.append({"group": group, "n_families": count})

branch_counts = pd.DataFrame(rows)

# Check that the groups account for the entire candidate dataset.
total = int(family_count.iloc[0]["n_families"])
assert branch_counts["n_families"].sum() == total, "Counts do not match."

branch_counts["share_pct"] = (
    100 * branch_counts["n_families"] / total
).round(1)

branch_counts

,group,n_families,share_pct
0,CPC only,2217,46.7
1,Keywords only,2472,52.1
2,Both CPC and keywords,60,1.3


## Preliminary check: CPC-only families

We randomly select 20 families retrieved through A01B79/005.

We inspect a representative title and abstract for each family to check:
- whether the invention fits our crop-production scope;
- how the technology is described;
- whether missing text or wording differences explain the keyword mismatch.

This is an exploratory check, not the final validation sample.

In [8]:
from sqlalchemy import case

# Sample from CPC-only families within our analysis period.
cpc_only_query = (
    db.query(scope.c.docdb_family_id)
    .select_from(scope)
    .filter(has_cpc, ~has_keywords)
    .order_by(scope.c.docdb_family_id)
)

cpc_only_ids = patstat.df(cpc_only_query)

sample_ids = cpc_only_ids.sample(
    n=min(20, len(cpc_only_ids)),
    random_state=38,
)["docdb_family_id"].tolist()

# Retrieve English text where available, keeping families without it.
text_query = (
    db.query(
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_id,
        TLS201_APPLN.appln_filing_date,
        func.max(
            case(
                (
                    func.lower(func.trim(
                        TLS202_APPLN_TITLE.appln_title_lg
                    )) == "en",
                    TLS202_APPLN_TITLE.appln_title,
                ),
                else_=None,
            )
        ).label("title"),
        func.max(
            case(
                (
                    func.lower(func.trim(
                        TLS203_APPLN_ABSTR.appln_abstract_lg
                    )) == "en",
                    TLS203_APPLN_ABSTR.appln_abstract,
                ),
                else_=None,
            )
        ).label("abstract"),
    )
    .select_from(TLS201_APPLN)
    .outerjoin(
        TLS202_APPLN_TITLE,
        TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id,
    )
    .outerjoin(
        TLS203_APPLN_ABSTR,
        TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id,
    )
    .filter(TLS201_APPLN.docdb_family_id.in_(sample_ids))
    .group_by(
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_id,
        TLS201_APPLN.appln_filing_date,
    )
)

texts = patstat.df(text_query)

# Prefer an application with an English abstract, then an English title.
texts["has_abstract"] = texts["abstract"].fillna("").str.strip().ne("")
texts["has_title"] = texts["title"].fillna("").str.strip().ne("")

inspection = (
    texts.sort_values(
        [
            "docdb_family_id",
            "has_abstract",
            "has_title",
            "appln_filing_date",
            "appln_id",
        ],
        ascending=[True, False, False, True, True],
    )
    .drop_duplicates("docdb_family_id")
    [["docdb_family_id", "appln_id", "title", "abstract"]]
    .reset_index(drop=True)
)

# Display the first five without truncating their text.
for _, row in inspection.head(5).iterrows():
    print(f"\nFAMILY: {row['docdb_family_id']}")
    print(f"APPLICATION: {row['appln_id']}")
    print("TITLE:", row["title"])
    print("ABSTRACT:", row["abstract"])
    print("-" * 80)


FAMILY: 8859729
APPLICATION: 15940158
TITLE: METHOD FOR PROCESSING GEOREFERENCED ELECTRICAL RESISTIVITY MEASUREMENTS FOR THE REAL-TIME ELECTRICAL MAPPING OF SOIL
ABSTRACT: The invention relates to a method for the georeferenced measuring of electrical resistivity for the electrical mapping of soil. The area of soil to be mapped is divided up into a fine network of points. According to the invention, measuring means are used to obtain n parameters (n being at least equal to 3) characterising the electrical resistivity at a given point at n different depths. In addition, an absolute position measurement and k relative displacement measurements are obtained using positioning means for a number k of points measured. The three measurements obtained are subsequently sent to a microcontroller which synchronises the acquisitions. The data sent by the microcontroller are digitally processed in real time on a computer. Furthermore, two different display windows are used to view, simultaneously 